# 08a — Thu thập MMKG di sản – danh thắng Việt Nam (ViHeritage-MMKG)

Cấp độ 3, bước dữ liệu. Notebook này lấy thực thể, triple, mô tả và ảnh từ Wikidata, Wikimedia Commons và Wikipedia tiếng Việt,
rồi xuất phiếu kiểm tra 100 ảnh cho 2 người chấm. Chưa chia split; việc đó ở `08b`.

| Bước | Nội dung | Mục kế hoạch |
|---|---|---|
| A0 | Cấu hình, HTTP có cache (chạy lại không gọi mạng) | 4 |
| A1 | Kiểm tra độ phủ: đủ ≥ 80 di tích/danh thắng có nhãn vi + Wikipedia vi + ảnh thì mới đi tiếp | 4.0 |
| A2 | Xếp hạng ứng viên | 4.1 |
| A3 | Ảnh: P18 + category Commons (P373), lọc giấy phép/kích thước/từ khóa, tải thumbnail 960 px | 4.4 |
| A4 | Lọc CLIP zero-shot (bản đồ, tài liệu, chân dung…), khử trùng, chọn tối đa 6 ảnh/thực thể | 4.4 |
| A5 | Chốt thực thể có ảnh; thêm thực thể không ảnh (tỉnh, nhân vật, khái niệm) | 4.1 |
| A6 | Triple (statement truthy), tỉnh theo địa giới 2025 | 4.2 |
| A7 | Văn bản: nhãn, bí danh, mô tả, mở bài Wikipedia vi | 4.3 |
| A8 | Phiếu kiểm tra 100 ảnh + lưu file + kiểm quy mô theo đề | 4.5 |

**Chạy:** Colab (CPU đủ, GPU nhanh hơn ở A4) → Run all. Cần Internet. Lần chạy đầu mất khoảng 40–60 phút, phần lớn là tải ảnh
với tốc độ giới hạn để đúng chính sách của Wikimedia. Mọi phản hồi API được lưu vào `raw/http_cache.tar.gz` và ảnh ứng viên vào
`raw/commons_candidate_images.tar` trên Drive, nên chạy lại không cần mạng và cho đúng kết quả cũ, kể cả khi Wikidata đã đổi.
Muốn lấy dữ liệu mới thì đặt biến môi trường `MMKG_REFRESH=1`.

**Sau khi chạy xong:** gửi file `review/image_check_raterA.csv` và `image_check_raterB.csv` cho 2 người chấm (xem A8).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 1, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 2, globals())


## A1. Kiểm tra độ phủ (mục 4.0)

Ứng viên lấy theo hai đường, không gõ tay mã QID của loại công trình:
1. Mọi mục ở Việt Nam có **danh hiệu di sản** (P1435).
2. Mục ở Việt Nam thuộc các **loại** (P31) phổ biến có ảnh và bài Wikipedia vi, trong đó chỉ giữ loại có nhãn khớp danh sách
   từ khóa công trình/danh thắng. Danh sách loại đã chọn được in ra và lưu vào `raw/class_whitelist.csv` để kiểm tra.

Một mục đạt khi có nhãn tiếng Việt, bài Wikipedia tiếng Việt, và ảnh (P18 hoặc category Commons P373).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 4, globals())


## A2. Xếp hạng ứng viên (mục 4.1)

Xếp theo số sitelink (mức độ nổi tiếng, giảm bớt mục ít ảnh/ít thông tin), hòa thì theo số statement. Giữ tối đa
`N_NATURAL_MAX` danh thắng thiên nhiên. Lấy dư `N_CANDIDATES` mục để bù những mục không đủ ảnh ở A3–A4.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 6, globals())


## A3. Ảnh từ Commons (mục 4.4)

Với mỗi ứng viên: file của P18 (`link_method = P18`), file trong category P373 (`P373_d0`), và nếu độ sâu 0 có ít hơn 4 ảnh đạt
thì thêm file trong các category con (`P373_d1`, bỏ category con về người, bản đồ, tài liệu).
Lọc trước khi tải: JPEG/PNG; cạnh ngắn ≥ 400 px; tỉ lệ cạnh ≤ 2,5; giấy phép CC0 / phạm vi công cộng / CC BY / CC BY-SA;
tên file và category không chứa từ khóa bản đồ, sơ đồ, logo, cờ, tem, tài liệu, chân dung.
Lý do loại của từng file được lưu trong `raw/commons_candidates.csv`.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 8, globals())


## A4. Lọc bằng CLIP, khử trùng, chọn ảnh (mục 4.4)

CLIP ViT-B/32 (cùng bộ mã hóa ảnh dùng cho đặc trưng ở `08b`) so mỗi ảnh với nhãn "ảnh công trình/cảnh quan" và nhãn
"bản đồ, mặt bằng, tài liệu, logo, cờ, tem, chân dung, nhóm người, đồ ăn, ảnh chụp màn hình, tranh vẽ". Bỏ ảnh có tổng xác suất
nhóm thứ hai ≥ 0,5. Khử trùng: SHA-256 trùng hệt, hoặc cosine CLIP ≥ 0,95 (trong một thực thể giữ ảnh đầu; giữa hai thực thể
bỏ cả hai). Mỗi thực thể giữ tối đa 6 ảnh: ảnh P18 trước, rồi chọn lần lượt ảnh khác nhất với các ảnh đã chọn.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 10, globals())


## A5. Chốt thực thể có ảnh, thêm thực thể không ảnh (mục 4.1)

Lấy theo thứ tự xếp hạng ở A2: ưu tiên thực thể có ≥ 3 ảnh, rồi đến thực thể có 2 ảnh, tới khi đủ 80–88 thực thể và tổng ảnh
không quá `TOTAL_IMG_MAX`; tối đa 15 danh thắng thiên nhiên, 6 cây cầu, 8 bảo tàng. Thực thể không ảnh là đầu kia của các quan hệ
ở mục 3.2, chỉ giữ đầu xuất hiện ≥ 2 lần, riêng tỉnh luôn giữ. Nhân vật chỉ giữ người đã mất (có ngày mất P570).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 12, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 13, globals())


## A6. Triple (mục 4.2)

Statement truthy (hạng tốt nhất, như `wdt:`), cả hai đầu phải nằm trong tập thực thể. Tỉnh lấy theo chuỗi P131 rồi đổi sang
34 tỉnh mới. Quan hệ giáp ranh giữa tỉnh mới: hai tỉnh mới giáp nhau khi có hai tỉnh cũ thuộc chúng giáp nhau (P47), lưu cả
hai chiều. Thêm P53 (dòng họ, triều đại) cho nhân vật, P279 giữa các khái niệm, P527 đảo chiều thành "thuộc quần thể".

Hai quan hệ **suy ra** từ dữ liệu Wikidata (không phải statement có sẵn; cột `source` ghi rõ, tắt được bằng cờ ở A0):
- `NEARBY` "ở gần": từ tọa độ P625, mỗi thực thể có ảnh nối với 2 thực thể có ảnh gần nhất trong vòng 5 km, lưu cả hai chiều.
- `CENTURY` "xây dựng vào": từ năm khởi lập P571 (độ chính xác ≥ thế kỷ), ra thực thể "thế kỷ N".
Cả hai suy ra từng thực thể một, không dùng thống kê trên tập train/test, nên không gây rò rỉ (mục 6 của đề).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 15, globals())


## A7. Văn bản (mục 4.3)

Nhãn, bí danh, mô tả ngắn tiếng Việt và tiếng Anh từ Wikidata; 2–3 câu đầu phần mở bài Wikipedia tiếng Việt
(`prop=extracts&exintro&explaintext`), lưu kèm `lastrevid` để ghi nguồn đúng phiên bản (CC BY-SA).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 17, globals())


## A8. Lưu file, kiểm quy mô, phiếu kiểm tra 100 ảnh (mục 4.5)

**Việc tay 1 — kiểm tra liên kết ảnh – thực thể.** 100 ảnh chọn ngẫu nhiên, chia tầng theo `link_method`. Hai người chấm
**độc lập** (không xem bài của nhau), người chấm không phải người viết bộ lọc ảnh:
1. Mở `review/image_check.html` (tải về máy rồi mở bằng trình duyệt) để xem ảnh kèm tên, mô tả thực thể.
2. Điền cột `label` trong file của mình (`image_check_raterA.csv` hoặc `image_check_raterB.csv`, mở bằng Google Sheets/Excel):
   `dung` = ảnh đúng là thực thể này; `sai` = ảnh của nơi khác hoặc không chụp thực thể; `khong_ro` = không đủ thông tin để biết.
3. Lưu lại đúng tên file, đúng thư mục `review/`. Notebook `08b` đọc hai file này để tính độ chính xác và Cohen κ.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 19, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08a_mmkg_collect', 20, globals())
